# Universal fd_cache generator (Kaggle GPU)

Generate FontDiffusion handwritten-style images for **every chu-Nom character** in the NomNaTong font CJK ranges (~21,800 chars). One-time run — produces a permanent cache that any future book can reuse without re-generating on Kaggle/Colab.

**Hardware**: Kaggle **GPU T4** (Settings → Accelerator → GPU T4 x2). P100 is faster but Kaggle's PyTorch 2.10 dropped Pascal (sm_60) support, so P100 fails. Time budget on T4: **~10-12 h, may need 2 sessions**.

**Resume-able**: every 500 chars, the notebook pushes the current state to a HuggingFace Hub repo. If Kaggle resets the session, just restart the notebook — cell 5 will re-download the partial cache and skip already-generated chars.

**Output**: `mdnt571/gannhanocr-universal-fd-cache` containing 21,837 PNGs named `U+XXXX.png`. Final size ~1.5 GB.

**Style strategy**: 1 representative crop from CacThanhTruyen2 — a clean, well-scanned book — used as the style reference for all chars. This produces a consistent style across the whole cache.

## 1. Verify GPU + install deps

Check that GPU is enabled (notebook should already be set to GPU P100 in Settings).

Adds Kaggle-specific bits: secrets for `HF_TOKEN`, `tqdm` for progress.

In [ ]:
!nvidia-smi || echo '⚠️  No GPU. Settings → Accelerator → GPU T4.'

# IMPORTANT: FontDiffusion pins very specific versions in its pyproject.toml:
#   diffusers==0.36.0, torch==2.9.1, accelerate==1.12.0
# Kaggle ships PyTorch 2.10 + would auto-pull diffusers latest. Mismatch causes
# the DPM-Solver scheduler API drift → output is pure noise. Pin everything.

import subprocess
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
print(f'GPU detected: {gpu_name}')

# Downgrade torch to 2.9.1 (matches FontDiffusion pyproject pin).
# Use cu121 wheels — works on T4 (sm_75). P100 (sm_60) still unsupported, use T4.
%pip install -q torch==2.9.1 torchvision==0.24.1 \
    --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

# Pin all FontDiffusion deps to versions in font_diffusion/pyproject.toml
%pip install -q \
    diffusers==0.36.0 \
    accelerate==1.12.0 \
    safetensors==0.7.0 \
    einops==0.8.1 \
    kornia==0.8.2 \
    info-nce-pytorch==0.1.4 \
    fonttools==4.61.1 \
    pygame==2.6.1 \
    scikit-image==0.26.0 \
    scipy==1.17.0 2>&1 | tail -3

print('\n⚠️  If torch was downgraded, RESTART the kernel before running the next cell.')
print('   (Runtime → Restart, then re-run this cell — it will be a no-op the second time.)\n')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    sm = f'sm_{cap[0]}{cap[1]}'
    ok = sm in supported
    print(f'  GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
    print(f'  Compute capability: {sm}  → {"✓ supported" if ok else "✗ NOT in supported list " + str(supported)}')
    if not ok:
        print('\n🛑 GPU not compatible with this PyTorch. Switch to T4 in Settings.')

import diffusers
print(f'diffusers {diffusers.__version__}  (must be 0.36.0)')
assert diffusers.__version__ == '0.36.0', \
    f'diffusers version mismatch! Got {diffusers.__version__}, need 0.36.0. Restart kernel and re-run.'


## 2. Configure HuggingFace Hub

We push the partial cache to a HF Hub repo every 500 chars so a Kaggle session reset never wipes progress. Create the repo once (any name) and add `HF_TOKEN` to Kaggle Secrets (Add-ons → Secrets).

Edit `HF_REPO` below to your repo (will be created if missing).

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, create_repo

# ════════════════════════════════════════════════════════════════════
# Your HuggingFace Hub repo for the universal fd_cache.
# This will be auto-created if it doesn't exist.
# Format: '<hf-username>/<repo-name>'
# ════════════════════════════════════════════════════════════════════
HF_REPO = 'mdnt571/gannhanocr-universal-fd-cache'
HF_REPO_TYPE = 'dataset'
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')  # add HF_TOKEN to Kaggle Add-ons → Secrets
# ════════════════════════════════════════════════════════════════════

login(token=HF_TOKEN, add_to_git_credential=False)

# Verify token + show identity
api = HfApi()
me = api.whoami()
print(f'✓ Logged in as: {me["name"]}')

expected_user = HF_REPO.split('/')[0]
if me['name'] != expected_user:
    print(f'⚠️  Token belongs to "{me["name"]}" but HF_REPO uses "{expected_user}".')
    print(f'    Either change HF_REPO to "{me["name"]}/..." or use a token from "{expected_user}".')

# Create dataset repo if missing
try:
    create_repo(repo_id=HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    print(f'✓ HF repo ready: https://huggingface.co/datasets/{HF_REPO}')
except Exception as e:
    print(f'⚠️  create_repo failed: {e}')
    print('   If your token is fine-grained, ensure it has:')
    print('     • Write access to repos under your username')
    print('     • "Create new repos" enabled')
    print('   Easiest fix: create a token of type "Write" instead of "Fine-grained".')

## 3. Clone the GanNhanOCR repo (with submodule)

Pulls main repo + `font_diffusion` submodule (FontDiffusion code + fonts).

In [3]:
import os, sys, shutil
from pathlib import Path

REPO_URL = 'https://github.com/truong571/GanNhanOCR.git'
PROJECT_ROOT = Path('/kaggle/working/GanNhanOCR')

def _populated(root: Path) -> bool:
    return (root / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf').exists()

if PROJECT_ROOT.exists():
    !cd {PROJECT_ROOT} && git pull --ff-only && git submodule update --init --recursive
    if not _populated(PROJECT_ROOT):
        # Stale empty submodule — wipe and re-init
        shutil.rmtree(PROJECT_ROOT / 'font_diffusion', ignore_errors=True)
        !cd {PROJECT_ROOT} && git submodule update --init --recursive
else:
    !git clone --recurse-submodules --depth 1 {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

assert _populated(PROJECT_ROOT), 'Submodule clone failed — fonts not present.'
print(f'✓ Repo at {PROJECT_ROOT}')
!ls font_diffusion/fonts/ | head -5

Cloning into '/kaggle/working/GanNhanOCR'...
remote: Enumerating objects: 123400, done.
remote: Counting objects: 100% (123400/123400), done.
remote: Compressing objects: 100% (122006/122006), done.
remote: Total 123400 (delta 1402), reused 123302 (delta 1388), pack-reused 0 (from 0)
Receiving objects: 100% (123400/123400), 319.86 MiB | 42.58 MiB/s, done.
Resolving deltas: 100% (1402/1402), done.
Updating files: 100% (123970/123970), done.
Submodule 'font_diffusion' (https://github.com/dzungphieuluuky/FontDiffusion.git) registered for path 'font_diffusion'
Cloning into '/kaggle/working/GanNhanOCR/font_diffusion'...
remote: Enumerating objects: 25434, done.        
remote: Counting objects: 100% (855/855), done.        
remote: Compressing objects: 100% (139/139), done.        
remote: Total 25434 (delta 786), reused 746 (delta 716), pack-reused 24579 (from 4)        
Receiving objects: 100% (25434/25434), 317.43 MiB | 35.92 MiB/s, done.
Resolving deltas: 100% (3928/3928), done.
Submodu

## 4. Download FontDiffusion checkpoints from HF Hub

Pulls the 7 component files (unet, encoders, projections) into both DRO/ and FST/ checkpoint dirs so the loader finds them.

In [ ]:
from huggingface_hub import hf_hub_download

# Use the PRODUCTION weights at the HF repo root (fully trained, non-FST).
# The /FST/checkpoint_step_1500/ subfolder is a training snapshot at step 1500
# and produces near-noise output — do NOT use it.
# Production root only has unet/style_encoder/content_encoder; we run with
# use_fst=False (already set in core/ranking/fontdiffusion_gen.py).

FD_HF = 'dzungpham/font-diffusion-weights'
CKPT_DIR = PROJECT_ROOT / 'font_diffusion' / 'ckpt' / 'PROD'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TO_FETCH = [
    'unet.safetensors',
    'style_encoder.safetensors',
    'content_encoder.safetensors',
]
for fn in TO_FETCH:
    dst = CKPT_DIR / fn
    if not dst.exists():
        cached = hf_hub_download(repo_id=FD_HF, filename=fn)  # ROOT, not FST/...
        shutil.copy2(cached, dst)
    print(f'  ✓ {fn}  ({dst.stat().st_size / 1024 / 1024:.1f} MB)')

# Wrapper still expects phase1_ckpt_dir for FST path; point it at the same dir
# (use_fst=False so it won\'t actually load FST modules).
FST_CKPT_DIR = CKPT_DIR

print(f'\n✓ FontDiffusion production weights ready at {CKPT_DIR}')


## 5. Resume from previous run (if any)

Pulls existing PNGs from the HF Hub repo into the local cache dir. If this is the first run, no-op.

In [5]:
from huggingface_hub import snapshot_download

CACHE_DIR = Path('/kaggle/working/_universal_fd_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Pulling existing cache from {HF_REPO} ...')
try:
    snapshot_download(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        local_dir=str(CACHE_DIR),
        allow_patterns='U+*.png',  # only PNGs, skip README etc.
    )
except Exception as e:
    print(f'  (no remote cache yet: {type(e).__name__})')

existing = sorted(CACHE_DIR.glob('U+*.png'))
print(f'✓ Resume state: {len(existing)} PNGs already on remote (will skip)')

Pulling existing cache from mdnt571/gannhanocr-universal-fd-cache ...


Fetching 0 files: 0it [00:00, ?it/s]

✓ Resume state: 0 PNGs already on remote (will skip)


## 6. Build the work list

Read `kaggle_diffusion/exports/char_universe.txt` (21,837 chars). Filter out chars already in the cache. The remaining list is what this session needs to generate.

In [6]:
universe_file = PROJECT_ROOT / 'kaggle_diffusion' / 'exports' / 'char_universe.txt'
with open(universe_file, encoding='utf-8') as f:
    all_chars = [line.strip() for line in f if line.strip()]

done_codepoints = {p.stem for p in existing}  # 'U+4E8C' etc.

todo = []
for c in all_chars:
    cp = f'U+{ord(c):04X}'
    if cp not in done_codepoints:
        todo.append(c)

print(f'Total universe:  {len(all_chars):>6,} chars')
print(f'Already done:    {len(done_codepoints):>6,}')
print(f'To generate:     {len(todo):>6,} chars')
if not todo:
    print('\n🎉 Nothing to do — cache complete!')

Total universe:  21,837 chars
Already done:         0
To generate:     21,837 chars


## 7. Load FontDiffusion pipeline (one time)

In [ ]:
from core.ranking.fontdiffusion_gen import FontDiffusionGenerator

print('Loading FontDiffusion (~30 s)...')
generator = FontDiffusionGenerator(
    ckpt_dir=str(CKPT_DIR),
    phase1_ckpt_dir=str(FST_CKPT_DIR),  # unused when use_fst=False but harmless
    font_path=str(PROJECT_ROOT / 'font_diffusion' / 'fonts' / 'NomNaTong-Regular.ttf'),
    cache_dir=str(CACHE_DIR),
)
print('✓ Loaded')


## 7.5 Sanity test — generate 5 sample chars

Before committing to a 6-8 h run, generate 5 representative chars and verify the
output looks reasonable. This is fast (~30 s) and catches:
- Broken style image (empty / wrong format)
- FontDiffusion config mismatch
- GPU OOM at this batch size
- Per-char generator errors

If the output thumbnails look like handwritten chu-Nom (not noise / blank), proceed
to cell 8. Otherwise, fix the issue and re-run this cell.

In [ ]:
import time
import numpy as np
import cv2
from PIL import Image
from IPython.display import display

# Tuned defaults:
#   SachThanhTruyen2  → thinner strokes (printed-book style)
#   guidance_scale=2  → minimum reasonable style strength (1.5 starts losing structure)
#   ERODE_ITERS=2     → morphological erosion of ink, ~60% thinner strokes to match style ref
STYLE_BOOK = 'SachThanhTruyen2'
GUIDANCE_SCALE = 2.0
ERODE_ITERS = 2

STYLE_IMAGE_PATH = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_BOOK}.png')
assert Path(STYLE_IMAGE_PATH).exists(), f'Style image missing: {STYLE_IMAGE_PATH}'
print(f'Style: {STYLE_IMAGE_PATH}')
display(Image.open(STYLE_IMAGE_PATH).resize((128, 128)))

generator._load_pipeline()
generator.pipe.guidance_scale = GUIDANCE_SCALE
print(f'guidance_scale = {generator.pipe.guidance_scale}  |  erode_iters = {ERODE_ITERS}')


def erode_strokes(img_path, iters):
    """Thin black ink on white bg via morphological dilation of background."""
    arr = np.array(Image.open(img_path).convert('L'))
    if iters > 0:
        kernel = np.ones((3, 3), np.uint8)
        arr = cv2.dilate(arr, kernel, iterations=iters)
    Image.fromarray(arr).save(img_path)


# Pick 5 well-known chu-Nom chars for sanity test
TEST_CHARS = ['一', '二', '三', '人', '月']
print(f'\nGenerating {len(TEST_CHARS)} sanity-test chars: {TEST_CHARS}')

t0 = time.time()
generator.generate(TEST_CHARS, STYLE_IMAGE_PATH, style_name='_sanity')
print(f'  Time: {(time.time()-t0):.1f}s ({(time.time()-t0)/len(TEST_CHARS):.1f}s/char)')

# Apply erosion to thin strokes
sanity_dir = CACHE_DIR / '_sanity'
for c in TEST_CHARS:
    p = sanity_dir / f'U+{ord(c):04X}.png'
    if p.exists():
        erode_strokes(p, ERODE_ITERS)

print(f'\nGenerated images (post-erosion):')
for c in TEST_CHARS:
    img_path = sanity_dir / f'U+{ord(c):04X}.png'
    if img_path.exists():
        print(f'  ✓ {c} → U+{ord(c):04X}.png')
        display(Image.open(img_path).resize((96, 96)))
    else:
        print(f'  ✗ {c} → not generated')

# Cleanup sanity files (don\'t pollute final cache)
import shutil as _sh
if sanity_dir.exists():
    _sh.rmtree(sanity_dir)
print('\n✓ Sanity test complete. Inspect the images above. If they look like handwritten chu-Nom, proceed.')


## 8. Generate + periodic checkpoint upload

Loop: generate in chunks of `CHUNK`, then upload each completed chunk to HF Hub. Set `CHUNK = 500` (every ~10 min on P100, balance between upload overhead and crash recovery).

In [ ]:
import time
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm

CHUNK = 500
STYLE_BOOK = 'SachThanhTruyen2'
GUIDANCE_SCALE = 2.0
ERODE_ITERS = 2  # post-process to thin strokes; 0 = off, 1 = ~30% thinner, 2 = ~60% thinner
STYLE_IMAGE = str(PROJECT_ROOT / 'kaggle_diffusion' / 'style_references' / f'{STYLE_BOOK}.png')

assert Path(STYLE_IMAGE).exists(), f'Style image missing: {STYLE_IMAGE}'

generator._load_pipeline()
generator.pipe.guidance_scale = GUIDANCE_SCALE
print(f'STYLE_BOOK = {STYLE_BOOK}  |  guidance_scale = {GUIDANCE_SCALE}  |  erode_iters = {ERODE_ITERS}')


def _erode_one(p):
    arr = np.array(Image.open(p).convert('L'))
    if ERODE_ITERS > 0:
        arr = cv2.dilate(arr, np.ones((3, 3), np.uint8), iterations=ERODE_ITERS)
    Image.fromarray(arr).save(p)


def upload_chunk():
    """Push current cache state to HF Hub. Skips files already on remote."""
    n = sum(1 for _ in CACHE_DIR.glob('U+*.png'))
    print(f'  ⏫ Uploading {n:,} PNGs to {HF_REPO}...', flush=True)
    api.upload_folder(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        folder_path=str(CACHE_DIR),
        allow_patterns='U+*.png',
        commit_message=f'add chunk — {n:,} chars total',
    )
    print(f'  ✓ Pushed.', flush=True)


# Generator stores PNGs under cache_dir/<style_name>/U+XXXX.png by default.
# We want them flat in CACHE_DIR (so resume snapshot_download works on flat layout).
# Override style_name to '' so subdir is empty? Actually the generate() method
# expects a non-empty style_name for sub-cache. We use 'universal' subfolder,
# then move PNGs up at end of each chunk.

STYLE_NAME = 'universal'
sub = CACHE_DIR / STYLE_NAME
sub.mkdir(parents=True, exist_ok=True)


def flatten():
    """Move PNGs from CACHE_DIR/universal/ to CACHE_DIR/ for flat layout on HF Hub."""
    for p in sub.glob('U+*.png'):
        dst = CACHE_DIR / p.name
        if not dst.exists():
            _erode_one(p)  # thin strokes before final placement
            shutil.move(str(p), str(dst))


if todo:
    t0 = time.time()
    pbar = tqdm(total=len(todo), desc='Generate', unit='char', dynamic_ncols=True)
    for i in range(0, len(todo), CHUNK):
        batch = todo[i:i + CHUNK]
        try:
            generator.generate(batch, STYLE_IMAGE, style_name=STYLE_NAME)
        except Exception as e:
            print(f'  ⚠️  chunk {i}–{i+len(batch)}: {type(e).__name__}: {e}')
            # still try to upload whatever was generated
        flatten()
        pbar.update(len(batch))
        # Push to HF after each chunk
        upload_chunk()
        # Free GPU memory between chunks
        torch.cuda.empty_cache()
    pbar.close()
    print(f'\n✓ Generation done in {(time.time()-t0)/60:.1f} min')
else:
    print('Nothing to generate.')

## 9. Final state + verification

After generation finishes (or session resets), confirm cache size matches universe.

In [ ]:
final = sorted(CACHE_DIR.glob('U+*.png'))
size_mb = sum(p.stat().st_size for p in final) / 1024 / 1024
print(f'Final cache: {len(final):,} PNGs / {len(all_chars):,} target')
print(f'Total size: {size_mb:.0f} MB')
print(f'Coverage: {len(final)/len(all_chars)*100:.1f}%')

if len(final) < len(all_chars):
    missing_codepoints = set(f'U+{ord(c):04X}' for c in all_chars) - {p.stem for p in final}
    print(f'\n{len(missing_codepoints)} chars still missing.')
    print('Restart this notebook to resume — cell 5 will pull state and continue.')

## 10. Use the cache from your Mac

Once 100% complete, on the Mac side:

```sh
cd /path/to/GanNhanOCR
huggingface-cli download mdnt571/gannhanocr-universal-fd-cache \
  --repo-type=dataset \
  --local-dir prepared/_universal_fd_cache/

./run_pipeline.sh --step 3 && ./run_pipeline.sh --step 4
```

Step 3 will use the universal cache for any chu-Nom char in any book — no more Colab/Kaggle generation runs needed.